In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import glob
import cv2

from scipy import stats

from skimage.measure import shannon_entropy
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Dataset Analysis and Signal Processing: SETI Radio Spectrograms

In the search for **extraterrestrial intelligence (SETI)**, raw radio signals captured by telescopes (such as the Allen Telescope Array) are converted into **two-dimensional spectrograms** (Time vs. Frequency). This exploratory data analysis (EDA) examines the structural, statistical, and spatial properties of a **7000-image SETI dataset from Kaggle** (https://www.kaggle.com/datasets/tentotheminus9/seti-data/data). Through signal processing, color-space mapping transformations, distribution analysis, and hypothesis testing, we uncover the physical characteristics of these technosignatures.

In [ ]:
# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context("notebook", font_scale = 1.1)

In [ ]:
# Define dataset paths
DATA_DIR = "../data/raw"
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VALID_DIR = os.path.join(DATA_DIR, 'valid')
TEST_DIR = os.path.join(DATA_DIR, 'test')

In [ ]:
CLASSES = sorted([c for c in os.listdir(TRAIN_DIR) if not c.startswith('.')])
print(f"Environment initialized. Target classes detected: {CLASSES}")

## 1. Basic Dataset Expectations & Shape Verification
Before any modeling, we must verify the structural dimensions of our data. We are going to count how many images each split  has (Train, Validation, Test), verify class balance, and inspect the spatial dimensions (resolution and channel shape) of individual spectrogram images.

In [ ]:
def test_dataset_shapes_and_splits():
    # 1. Count splits
    splits = {'train': TRAIN_DIR, 'valid': VALID_DIR, 'test': TEST_DIR}
    split_totals = {}
    for split_name, split_path in splits.items():
        total = len(glob.glob(os.path.join(split_path, '**', '*.png'), recursive = True))
        split_totals[split_name] = total
        print(f"{split_name.lower()}: {total} images")
    print(f"Total Dataset Size: {sum(split_totals.values())} images\n")

    # 2. Check class balance & image shapes in Train set
    class_stats = []
    sample_shape = None
    
    for cls in CLASSES:
        cls_path = os.path.join(TRAIN_DIR, cls)
        img_paths = glob.glob(os.path.join(cls_path, '*.png'))
        count = len(img_paths)
        
        if count > 0:
            # Read sample image to check shape
            sample_img = cv2.imread(img_paths[0], cv2.IMREAD_UNCHANGED)
            sample_shape = sample_img.shape
            
        class_stats.append({'Class': cls, 'Count': count})

    df_classes = pd.DataFrame(class_stats)
    print("=== Training Class Balance ===")
    print(df_classes.to_string(index=False))
    print(f"\nNative Image Tensor Shape: {sample_shape} (Height, Width, Channels)")
    
    return df_classes

df_class_distribution = test_dataset_shapes_and_splits()

The total dataset consists of **7000 spectrogram images**, partitioned into **5,600 training samples (80%)**, **700 validation samples (10%)**, and **700 test samples (10%)**.

Every single one of the 7 technosignature classes (`brightpixel`, `narrowband`, `narrowbanddrd`, `noise`, `squarepulsednarrowband`, `squiggle`, and `squigglesquarepulsednarrowband`) contains **800 training images**. The dataset doesn’t have class imbalance.

The native image dimensions are **$384 \times 512$ pixels with 4 channels** `(384, 512, 4)`. The 4-channel configuration (typically RGBA or multi-layered data feeds common in radio astronomy waterfall plots) poses a limitation. Standard pre-trained Computer Vision backbones (such as ResNet50 or EfficientNet) accept **3-channel RGB tensors**. Therefore, our data pipeline must handle channel mapping or reduction (dropping the alpha channel or converting to a 3-channel layout) during tensor transformation. Furthermore, because native dimensions ($384 \times 512$) do not match standard CNN input requirements ($224 \times 224$), spatial resizing transformations are required.

## 2. Domain Knowledge: What Each Class Represents
These classes, based on SETI Institute definitions, represent the following:

In [ ]:
def plot_raw_class_samples():
    fig, axes = plt.subplots(1, 7, figsize = (22, 4))
    fig.suptitle('Raw Spectrogram Samples Across All 7 SETI Classes', fontsize = 18, y = 1.05)
    
    for i, cls in enumerate(CLASSES):
        img_path = glob.glob(os.path.join(TRAIN_DIR, cls, '*.png'))[5]
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        ax = axes[i]
        ax.imshow(img, cmap = 'gray')
        ax.set_title(cls, fontsize = 10, pad = 8)
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()

plot_raw_class_samples()

1. **Brightpixel:** A sudden, highly intense burst of energy confined to a tiny frequency band and time window.
2. **Narrowband:** A continuous signal at a single, unchanging frequency. It appears as a straight, unbroken line.
3. **Narrowbanddrd:** A narrowband signal with **Doppler Drift**. The line is diagonal, representing the relative acceleration between Earth and the extraterrestrial transmitter.
4. **Noise:** Pure cosmic background static. It lacks any concentrated structure.
5. **Square Pulsed Narrowband:** A narrowband signal that turns on and off at regular intervals (like a metronome or Morse code), appearing as a dashed line.
6. **Squiggle:** A signal whose frequency drifts in a wavy, sinusoidal pattern, possibly due to a rotating transmitter or complex modulation.
7. **Squigglesquarepulsednarrowband:** A wavy, drifting signal that is also pulsing on and off.

## 3. Color-Space Transformations and Information Hiding
Human eyes are bad at distinguishing subtle variations in monochrome (grayscale) intensity, but highly sensitive to color gradients. 

We are going to take a signal **(squiggle)** and render it using different matplotlib colormaps (`gray`, `magma`, `viridis`, `jet`). **Grayscale images** can "hide" low-amplitude signal tracks within background noise because the dynamic range is compressed for human viewing. **Colormaps** like `magma` or `viridis` highlight hidden frequency structures.

In [ ]:
def test_color_space_transformations():
    # A sample squiggle image
    sample_path = glob.glob(os.path.join(TRAIN_DIR, 'squiggle', '*.png'))[7]
    img = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)
    
    colormaps = ['gray', 'magma', 'viridis', 'jet', 'inferno']
    
    fig, axes = plt.subplots(1, len(colormaps), figsize = (20, 4))
    fig.suptitle('Perceptual Impact of Color Mapping on Signal Visibility (Squiggle Class)', fontsize = 16, y = 1.05)
    
    for i, cmap in enumerate(colormaps):
        ax = axes[i]
        im = ax.imshow(img, cmap = cmap)
        ax.set_title(f"Colormap: {cmap}", fontsize = 11)
        ax.axis('off')
        fig.colorbar(im, ax = ax, orientation = 'horizontal', fraction = 0.046, pad = 0.04)
        
    plt.tight_layout()
    plt.show()

test_color_space_transformations()

In the **gray** rendering, the faint, curving trajectory of the squiggle signal can easily blur into the background space noise because the human eye has a limited dynamic range for distinguishing shades of gray.

Perceptually uniform colormaps like **magma**, **viridis**, and **inferno** smoothly expand the color spectrum. They map low-intensity background noise to deep, dark tones (purples/blacks) and high-intensity technosignature tracks to glowing, high-contrast hues (yellows/oranges). This enhances the visual distinction of the signal's spatial curvature.

The **jet colormap** provides sharp color transitions (from blue to green to red), which can make high-amplitude peaks stand out instantly, though it is non-uniform and can sometimes introduce false visual boundaries.

## 4. Global Pixel Intensity & Dynamic Range Distribution
We are going to compute **pixel intensity histograms** for each class across the 0-255 range. This will reveal the underlying distribution of the background static versus signal power. A **narrow peak near 0** indicates a vacuum/quiet space background, while **heavy right-tails** represents high-power technosignatures.

In [ ]:
def test_pixel_intensity_distributions():
    plt.figure(figsize = (14, 6))
    
    for cls in CLASSES:
        paths = glob.glob(os.path.join(TRAIN_DIR, cls, '*.png'))[:50] # Sample 50 images per class
        pixels = []
        for p in paths:
            img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
            pixels.extend(img.flatten())
            
        sns.kdeplot(pixels, label = cls, fill = False, linewidth = 2)
        
    plt.title('Pixel Intensity Probability Density Functions (PDF) per Class', fontsize = 16)
    plt.xlabel('Pixel Intensity Value (0 - 255)')
    plt.ylabel('Density')
    plt.xlim(0, 255)
    plt.legend(bbox_to_anchor = (1.05, 1), loc = 'upper left')
    plt.tight_layout()
    plt.show()

test_pixel_intensity_distributions()

**All seven distribution curves** exhibit a sharp, towering peak clustered toward the lower end of the intensity spectrum (near $0$). This represents the vacuum/background space static. Because the vast majority of pixels in any given spectrogram represent empty space or ambient noise rather than signal tracks, the background overwhelmingly dictates the global pixel distribution.

While the curves follow a similar baseline path due to the shared background noise, **signal-bearing classes** (such as narrowband, squiggle, and squarepulsednarrowband) show gradual "tails" extending toward **higher intensity values** (bright regions) and the **noise class** drops off more steeply because it lacks **high-amplitude signal paths**.

From a model perspective, a **simple thresholding** will fails because the intensity curves overlap in the low-intensity range, it cannot separate signals from noise using a **simple global brightness threshold** (setting a rule like "if pixel value > 200, it's an alien signal"). Background noise artifacts frequently match or exceed the brightness of faint signals.

This intensity distribution proves that pixel values span a **wide variance** across images. In our PyTorch data pipeline, applying **image normalization** will scale these values into a uniform distribution, ensuring that gradients during backpropagation don't explode or vanish due to **raw brightness spikes**.

## 5. 2D Fast Fourier Transform (FFT) Frequency Analysis
Here, we are goint to convert spatial domain spectrograms into the frequency domain using a **2D FFT**. In radio astronomy, instrumental artifacts or systematic telescope noise often reveal **periodic grid patterns** in the frequency domain. This test checks whether our background noise is truly random or contaminated by hardware interference.

In [ ]:
def test_2d_fft_spectrum():
    fig, axes = plt.subplots(2, 2, figsize = (10, 10))
    fig.suptitle('Spatial vs Frequency Domain (2D FFT Magnitude Spectrum)', fontsize = 16)
    
    # Select one Noise image and one Narrowband image
    noise_img = cv2.imread(glob.glob(os.path.join(TRAIN_DIR, 'noise', '*.png'))[14], cv2.IMREAD_GRAYSCALE)
    signal_img = cv2.imread(glob.glob(os.path.join(TRAIN_DIR, 'narrowband', '*.png'))[32], cv2.IMREAD_GRAYSCALE)
    
    for idx, (img, name) in enumerate([(noise_img, 'Noise'), (signal_img, 'Narrowband Signal')]):
        # Compute 2D FFT
        f_transform = np.fft.fft2(img)
        f_shift = np.fft.fftshift(f_transform)
        magnitude = 20 * np.log(np.abs(f_shift) + 1)
        
        # Plot Original
        ax_orig = axes[idx, 0]
        ax_orig.imshow(img, cmap = 'magma')
        ax_orig.set_title(f"{name} (Spatial Domain)")
        ax_orig.axis('off')
        
        # Plot FFT
        ax_fft = axes[idx, 1]
        ax_fft.imshow(magnitude, cmap = 'magma')
        ax_fft.set_title(f"{name} (Frequency Domain - FFT)")
        ax_fft.axis('off')
        
    plt.tight_layout()
    plt.show()

test_2d_fft_spectrum()

The FFT magnitude spectrum of the **noise class** exhibits an **unstructured energy distribution** across the frequency plane. This confirms that the simulated background static conforms to **Gaussian noise** without instrumental grid artifacts or periodic hardware interference from the telescope array.

In contrast, the FFT spectrum of the **narrowband signal** displays sharp, concentrated harmonic spikes or cross-shaped intensity axes. 

This transformation highlights how technosignatures differ from background clutter. While noise is randomly dispersed across all spatial frequencies, structured transmissions concentrate energy into **frequency bins**.

## 6. Mean Class Templates & Time-Frequency Marginalization
In this section we will **average 200 images per class** to cancel out noise and reveal the "ground truth" signal template. Then, collapse the **2D matrix into 1D marginal profiles** across Time (X-axis) and Frequency (Y-axis). Individual spectrograms contain heavy noise. Averaging exposes the exact geometry of the signals. Marginal profiles prove whether energy is uniform across time or concentrated into precise pulses/frequencies.

In signal processing, when you overlay multiple independent images of the same signal class, two things happen:
- The signal has the same structural shape in every image. When you sum $N$ images, the signal grows linearly: $N \times \text{Signal}$;
- The background noise is random space static. When you sum $N$ random noise, they cancel each other out, but their total energy grows as the square root: $\sqrt{N} \times \text{Noise}$. Therefore, the SNR of your averaged template improves by a factor of $\sqrt{N}$.

In [ ]:
def test_mean_templates_and_marginalization():
    fig, axes = plt.subplots(3, 7, figsize = (24, 10))
    fig.suptitle('Ground Truth Mean Templates with Time & Frequency Marginal Profiles', fontsize = 18, y = 1.05)
    
    for i, cls in enumerate(CLASSES):
        paths = glob.glob(os.path.join(TRAIN_DIR, cls, '*.png'))[:200]
        accumulated = np.zeros((224, 224), dtype=np.float64)
        
        for p in paths:
            im = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
            accumulated += cv2.resize(im, (224, 224))
            
        mean_img = accumulated / len(paths)
        
        # Row 0: Mean Template Image
        axes[0, i].imshow(mean_img, cmap = 'magma')
        axes[0, i].set_title(cls, fontsize = 11)
        axes[0, i].axis('off')
        
        # Row 1: Time Profile (Mean across Y-axis)
        time_prof = np.mean(mean_img, axis = 0)
        axes[1, i].plot(time_prof, color = 'cyan', linewidth = 1.5)
        axes[1, i].set_title("Time Profile", fontsize = 9)
        axes[1, i].set_xticks([])
        axes[1, i].grid(True, alpha = 0.3)
        
        # Row 2: Frequency Profile (Mean across X-axis)
        freq_prof = np.mean(mean_img, axis = 1)
        axes[2, i].plot(freq_prof, color = 'lime', linewidth = 1.5)
        axes[2, i].set_title("Freq Profile", fontsize = 9)
        axes[2, i].set_xticks([])
        axes[2, i].grid(True, alpha = 0.3)

    plt.tight_layout()
    plt.show()

test_mean_templates_and_marginalization()

The **mean template images** confirm that each class maintains a spatial signature validating the structural consistency of the dataset.

The **1D time projections (cyan)** capture amplitude modulation. Continuous transmissions exhibit stable energy distributions over time, whereas pulsed signals (squarepulsednarrowband) demonstrate square-wave modulation peaks.

The **1D frequency projections (lime)** isolate spectral occupancy. Narrowband signals concentrate power into singular frequency bins, whereas frequency-drifting variants distribute energy across a broader spectral, underscoring the necessity of 2D convolutional filters that can capture spatial-frequency dependencies.

## 7. Machine Vision Edge Gradient Extraction (Simulating CNN Filters)
What we are going to do is appling **Gaussian smoothing** followed by **Sobel** and **Canny edge detection** operators. A CNN's early layers function identically to edge and gradient filters. This test shows that technosignatures possess sharp directional gradients that convolutional kernels can isolate from background.

In [ ]:
def test_edge_gradient_extraction():
    sample_path = glob.glob(os.path.join(TRAIN_DIR, 'narrowbanddrd', '*.png'))[3]
    img = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)
    blurred = cv2.GaussianBlur(img, (3, 3), 0)
    
    sobelx = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize = 3)
    sobely = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize = 3)
    sobel_combined = cv2.magnitude(sobelx, sobely)
    canny = cv2.Canny(blurred, 100, 200)

    fig, axes = plt.subplots(1, 4, figsize = (18, 4))
    fig.suptitle('Machine Vision Edge Extraction (Narrowband Doppler Drift)', fontsize = 16, y = 1.05)
    
    axes[0].imshow(img, cmap = 'gray'); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(sobelx, cmap = 'gray'); axes[1].set_title('Sobel X (Time Gradients)'); axes[1].axis('off')
    axes[2].imshow(sobely, cmap = 'gray'); axes[2].set_title('Sobel Y (Freq Gradients)'); axes[2].axis('off')
    axes[3].imshow(canny, cmap = 'gray'); axes[3].set_title('Canny Edge Map'); axes[3].axis('off')
    
    plt.tight_layout()
    plt.show()

test_edge_gradient_extraction()

1. **Panel 1: Original Grayscale Spectrogram (narrowbanddrd)**

The raw input image featuring a diagonal signal trace (Doppler drift) embedded within background space noise. The signal represents a shifting radio frequency over time due to relative orbital acceleration.

2. **Panel 2: Sobel X Operator (Time-Axis Gradients)**

By applying a horizontal derivative kernel, this filter isolates sharp changes in pixel intensity along the X-axis (Time). It highlights temporal transitions—the exact moments when the signal appears or shifts across time bins.

3. **Panel 3: Sobel Y Operator (Frequency-Axis Gradients)**

By applying a vertical derivative kernel, this filter isolates changes along the Y-axis (Frequency). It highlights the sharp boundaries of the frequency band where the technosignature is.

4. **Panel 4: Canny Edge Detection Map**

The Canny operator performs multi-stage processing (Gaussian smoothing, gradient magnitude calculation, non-maximum suppression, and hysteresis thresholding). The result is a clean, binarized edge map that isolates the continuous diagonal trajectory of the Doppler drift while canceling background noise.

## 8. Principal Component Analysis
Here, we will flatten images into 1D vectors, standardize features, apply PCA, and project the dataset onto 2 principal components. If the PCA plot shows a **massive overlapping blob**, it proves that the identifying characteristics are non-linear and spatial patterns will require deep feature extraction (CNNs).

In [ ]:
def test_pca_separability():
    X = []
    y = []
    
    for cls in CLASSES:
        paths = glob.glob(os.path.join(TRAIN_DIR, cls, '*.png'))[:100]
        for p in paths:
            img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
            resized = cv2.resize(img, (32, 32)) # Downsample to manage memory
            X.append(resized.flatten())
            y.append(cls)
            
    X = np.array(X)
    X_scaled = StandardScaler().fit_transform(X)
    
    pca = PCA(n_components = 2, random_state = 42)
    X_pca = pca.fit_transform(X_scaled)
    
    plt.figure(figsize = (10, 8))
    sns.scatterplot(x = X_pca[:, 0], y = X_pca[:, 1], hue = y, palette = 'tab10', s = 50, alpha = 0.8)
    plt.title(f'PCA Projection of SETI Spectrograms (Explained Variance: {sum(pca.explained_variance_ratio_) * 100:.2f}%)', fontsize = 14)
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0] * 100:.2f}%)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1] * 100:.2f}%)')
    plt.legend(bbox_to_anchor = (1.05, 1), loc = 'upper left')
    plt.tight_layout()
    plt.show()
    
test_pca_separability()

**All seven technosignature classes** converge into a overlapping multi-class shape without distinct linear boundaries. This demonstrates that global pixel-level variance and linear compression are insufficient to distinguish signal classes from background space noise.

Flattening 2D spectrograms into 1D vectors destroys the **spatial-temporal hierarchy** (such as frequency drift trajectories and time-dependent modulation). 

Because linear models and machine learning algorithms rely on linear decision boundaries, this PCA plot validates that non-linear feature extraction via CNNs is mandatory for achieving high classification accuracy.

## 9. Hypothesis Testing

### 9.1 Hypothesis 1: Sub-Class Spectral Dispersion (Narrowband vs. Narrowband DRD)

#### Scientific Context & Motivation
* **The Problem:** CNN frequently experience misclassifications between stationary **narrowband** and **narrowbanddrd** signals. Both classes display as sharp, high-intensity linear tracks, creating a visual overlap.
* **The Physical Difference:** A stationary narrowband signal remains locked within a constant frequency bin across time. In contrast, a narrowbanddrd traces a diagonal slope across the spectrogram due to relative orbital acceleration (Doppler drift). 

#### Formulating the Hypothesis
* **Objective:** To test whether Doppler-drifting signals exhibit a broader frequency dispersion profile than stationary signals.
* **Null Hypothesis ($H_0$):** There is no significant difference in mean spectral dispersion between narrowband and narrowbanddrd classes ($\mu_{\text{narrowband}} = \mu_{\text{drd}}$).
* **Alternative Hypothesis ($H_1$):** The narrowbanddrd exhibits a significantly greater frequency dispersion profile ($\mu_{\text{narrowband}} \neq \mu_{\text{drd}}$).

#### Methodology & Metric Selection
* **Metric:** **Frequency Axis Standard Deviation ($\sigma_y$)**. We isolate signal pixels using a 95th-percentile threshold and calculate the standard deviation of their vertical row indices. This quantifies how wide or sloped a signal's footprint is along the frequency axis.
* **Statistical Test:** **Welch’s t-test ($\alpha = 0.05$)**. 
* **Why this test?** Welch’s test doesn’t assume equal variances between groups. Because a straight horizontal line has a near-zero vertical spread while a sloping drift line has a wider spread, their variances differ.

#### Visualization Strategy
* **Method:** **Overlaid KDE Plot with Mean Reference Lines**.
* **Why this visualization?** An overlaid density curve shows the entire probability distribution and overlap range of the two classes. Adding vertical lines for the sample means provides an immediate visual representation of the central tendency shift between stationary and drifting transmissions.

In [ ]:
def compute_frequency_dispersion(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return 0.0
    _, thresh = cv2.threshold(img, np.percentile(img, 95), 255, cv2.THRESH_BINARY)
    y_indices, _ = np.nonzero(thresh)
    if len(y_indices) == 0:
        return 0.0
    return float(np.std(y_indices))

def test_h1_spectral_dispersion(data_dir = "../data/raw/train"):
    nb_files = glob.glob(os.path.join(data_dir, 'narrowband', '*.png'))[:200]
    drd_files = glob.glob(os.path.join(data_dir, 'narrowbanddrd', '*.png'))[:200]
    
    nb_disp = [compute_frequency_dispersion(p) for p in nb_files]
    drd_disp = [compute_frequency_dispersion(p) for p in drd_files]
    
    t_stat, p_val = stats.ttest_ind(nb_disp, drd_disp, equal_var = False)
    
    print("=== HYPOTHESIS 1 RESULTS: Spectral Dispersion ===")
    print(f"Mean Dispersion (Narrowband):     {np.mean(nb_disp):.4f}")
    print(f"Mean Dispersion (Narrowband DRD): {np.mean(drd_disp):.4f}")
    print(f"Welch's t-statistic:              {t_stat:.4f}")
    print(f"P-Value:                          {p_val:.4e}")
    print("Result:", "REJECT H0" if p_val < 0.05 else "FAIL TO REJECT H0")
    
    # DataFrame preparation for Seaborn KDE plot
    df_h1 = pd.DataFrame({
        'Dispersion': nb_disp + drd_disp,
        'Class': ['Narrowband'] * len(nb_disp) + ['Narrowband DRD'] * len(drd_disp)
    })
    
    # Overlaid KDE Density Plot with Mean Reference Lines
    plt.figure(figsize = (10, 5))
    sns.kdeplot(
        data = df_h1, 
        x = 'Dispersion', 
        hue = 'Class', 
        fill = True, 
        common_norm = False, 
        palette = "viridis", 
        alpha = 0.4, 
        linewidth = 2
    )
    
    # Add vertical lines for class means
    plt.axvline(np.mean(nb_disp), color = '#440154', linestyle = '--', linewidth = 1.5, label = f'Narrowband Mean ({np.mean(nb_disp):.1f})')
    plt.axvline(np.mean(drd_disp), color = '#21918c', linestyle = '--', linewidth = 1.5, label = f'Narrowband DRD Mean ({np.mean(drd_disp):.1f})')
    
    plt.title("H1: Probability Density of Spectral Dispersion (Narrowband vs. Narrowband DRD)", fontsize = 13, pad = 10)
    plt.xlabel("Frequency Axis Standard Deviation (Pixels)", fontsize = 11)
    plt.ylabel("Estimated Probability Density", fontsize = 11)
    plt.legend(loc = 'upper right', frameon = True)
    plt.tight_layout()
    plt.show()

test_h1_spectral_dispersion()

Because the resulting **p-value ($0.2917$)** is higher than our **threshold ($\alpha = 0.05$)**, there is no significant difference between the vertical standard deviations of these two sub-classes. The sample means are identical, proving that measuring the **vertical pixel spread alone is insufficient** to capture the difference between a straight vertical/horizontal line and a drift line.

Both stationary and drifting signals occupy a similar vertical span within the fixed-resolution coordinate grid of the spectrogram canvas. Because our metric computed the standard deviation of all active pixels without tracing directional trajectories, it registered a comparable pixel spread for both types. This explains why models like CNNs are challenged by these specific sub-classes. At a global level, their **macro-dispersions overlap completely**.

### 9.2 Hypothesis 2: Temporal Energy Distribution Symmetry (Center of Mass)

#### Scientific Context & Motivation
* **The Problem:** Technosignatures should maintain a balanced distribution across the observation window. If signal generation or padding introduces a front-loading or back-loading bias, models may learn positional artifacts instead of physical features.
* **The Physical Expectation:** A calibrated dataset of simulated radio tracks (narrowband, squiggle, and squarepulsednarrowband) should exhibit energy distributed across the time axis, centered around the midpoint.

#### Formulating the Hypothesis
* **Objective:** To verify whether the temporal center of mass for signal energy across active signal classes is centered at the exact midpoint ($0.5$ mark) of the observation window.
* **Null Hypothesis ($H_0$):** The mean normalized temporal center of mass for signal classes is equal to the midpoint of $0.5$ ($\mu = 0.5$).
* **Alternative Hypothesis ($H_1$):** The mean temporal center of mass is skewed away from the center ($\mu \neq 0.5$).

#### Methodology & Metric Selection
* **Metric:** **Normalized First-Moment Temporal Centroid ($C$)**. We collapse the 2D spectrogram matrix across the frequency axis (rows) to produce a 1D time profile (columns). Then calculate the intensity-weighted center of mass and normalize it by total image width to produce a value between $0.0$ (start) and $1.0$ (end).
* **Statistical Test:** **One-Sample t-test ($\alpha = 0.05$)**.
* **Why this test?** A one-sample t-test allows us to test whether the empirical mean of our sample distribution differs from a hypothesized population mean (popmean = 0.5).

#### Visualization Strategy
* **Method:** **Frequency Distribution Histogram with Reference Lines**.
* **Why this visualization?** A histogram provides a clear view of the spread of temporal centroids across hundreds of samples. Overlaying a dashed red line for the midpoint ($0.5$) and a orange line for the sample mean makes micro-biases immediately apparent.

In [ ]:
def compute_temporal_centroid(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return 0.5
    # Sum across frequency axis (rows) to get 1D time profile (columns)
    time_profile = np.sum(img, axis = 0)
    total_energy = np.sum(time_profile)
    if total_energy == 0:
        return 0.5
    x_coords = np.arange(len(time_profile))
    centroid = np.sum(x_coords * time_profile) / total_energy
    # Normalize by total width
    return centroid / img.shape[1]

def test_h3_temporal_symmetry(data_dir = "../data/raw/train"):
    # Aggregate across multiple signal classes to test overall dataset temporal symmetry
    target_classes = ['narrowband', 'squiggle', 'squarepulsednarrowband']
    all_centroids = []
    
    for cls in target_classes:
        files = glob.glob(os.path.join(data_dir, cls, '*.png'))[:200]
        for p in files:
            all_centroids.append(compute_temporal_centroid(p))
            
    t_stat, p_val = stats.ttest_1samp(all_centroids, popmean = 0.5)
    
    print("=== HYPOTHESIS 2 RESULTS: Temporal Center of Mass ===")
    print(f"Mean Temporal Centroid (Normalized):   {np.mean(all_centroids):.4f}")
    print(f"Theoretical Midpoint:                  0.5000")
    print(f"One-sample t-statistic:                {t_stat:.4f}")
    print(f"P-Value:                               {p_val:.4e}")
    print("Result:", "REJECT H0 (Skewed)" if p_val < 0.05 else "FAIL TO REJECT H0 (Symmetric)")
    
    plt.figure(figsize = (9, 5))
    sns.histplot(all_centroids, color = "teal", bins = 30)
    plt.axvline(0.5, color = 'red', linestyle = '--', linewidth = 2, label = 'Midpoint (0.5)')
    plt.axvline(np.mean(all_centroids), color = 'orange', linestyle = '-', linewidth = 2, label = f'Sample Mean ({np.mean(all_centroids):.3f})')
    plt.title("H2: Temporal Energy Center of Mass Distribution")
    plt.xlabel("Normalized Time Axis Centroid (0 = Start, 1 = End)")
    plt.ylabel("Frequency / Count")
    plt.legend()
    plt.tight_layout()
    plt.show()

test_h3_temporal_symmetry()

The **mean temporal centroid is $0.4990$**, which aligns nearly perfectly with the center of **$0.5000$**. Despite the proximity to the midpoint, the one-sample t-statistic (**$-30.4762$**) and near-zero p-value (**$6.7946 \times 10^{-124}$**) lead to the **rejection of the $H_0$**.

In **large samples ($N = 600$)**, standard error becomes very small, making even microscopic deviations significant. This confirms a slight, micro-skew in how signal padding or generation bounds are structured in the simulation pipeline. Because a **tiny bias exists**, our CNN will benefit from **data augmentations** (such as random time-shifting or horizontal flipping where applicable) to prevent the model from **overfitting** to exact center coordinates.